# Train with PPO + Domain Randomization

**Deviates from the reference repo's own structure on purpose**: their notebook redoes
a "balance only" -> "add position/yaw penalty" curriculum from scratch (phases 1-2)
before ever touching domain randomization, because their base training notebook used a
stripped-down reward. Ours (`02-balance-bot-with-ppo/uniwheel_env.py`) already trains
with position/yaw penalties on by default - so this notebook **warm-starts from stage
02's finished `best_model` checkpoint** instead of re-deriving what stage 02 already
did, and goes straight into the domain-randomization-specific phases:

1. Warm-start from stage 02's checkpoint (no training, just load + confirm it works
   in this new env class)
2. Observation noise + action delay
3. Motor noise + random pushes
4. Mass + friction randomization
5. Motor gain randomization
6. Ridge torque (**wheel axle only** - the reaction wheel never touches the ground,
   so it has no tire to hit a ridge with; see `uniwheel_env_dr.py`'s module docstring)

Each phase continues training the *same* agent from the previous phase (`agent=result.agent`),
same pattern as the reference repo - domain randomization is added incrementally so the
policy isn't hit with everything at once.


In [ ]:
# Import standard libraries
from dataclasses import replace
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
import numpy as np
import torch

# Import the DR-enabled environment
from uniwheel_env_dr import UniWheelEnv, DomainRandomConfig

# Add the folder containing the rl/ package to the path
sys.path.append("..")
from rl.ppo_trainer import PPOConfig, Agent, TrainResult, evaluate, train, \
    export_tb_plots_as_csv, export_actor_onnx


In [ ]:
# Settings
MJCF_PATH = Path("../../mechanical/urdf/uniwheel/uniwheel.xml")
SEED = 42
NUM_ENVS = 12                  # matches this machine's core count - see stage 02's README
                                # for why NUM_ENVS, not cuda, is the speed lever here
STEPS_PER_ENV = 500_000         # per phase - smaller than stage 02's since each phase is
                                # a refinement on an already-balancing policy, not learning
                                # from scratch

# Stage 02's finished checkpoint (best mean return 9970.5, independently verified with a
# full 10,000-step rollout - see workspace/software/02-balance-bot-with-ppo/README.md)
STAGE02_CHECKPOINT = Path(
    "../02-balance-bot-with-ppo/runs/UniWheel-v0__uniwheel-ppo__42__1788678088/best_model.cleanrl_model"
)


In [ ]:
# Configure PPO (same architecture/hyperparameters as stage 02, so the warm-started
# weights actually line up)
ppo_config = PPOConfig(
    exp_name = "uniwheel-ppo-dr",
    env_id = "UniWheel-v0",
    seed = SEED,
    cuda = False,                  # see stage 02's README - measured slower here, not a typo
    num_envs = NUM_ENVS,
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,
    num_steps = 2048,
    num_minibatches = 32,
    update_epochs = 10,
    anneal_lr = True,
    learning_rate = 3e-4,
    gamma = 0.99,
    gae_lambda = 0.95,
    clip_coef = 0.2,
    value_clip = 1.0,
    ent_coef = 0.0,
    vf_coef = 0.5,
    max_grad_norm = 0.5,
    checkpoint_interval = 20,
    save_model = True,
)


In [ ]:
def make_uniwheel_env(render, **kwargs):
    """Create one DR-capable uni wheel environment."""
    env = UniWheelEnv(
        mjcf_path=MJCF_PATH,
        render_mode="human" if render else None,
        **kwargs,
    )
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs uni wheel environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(lambda render=(i == 0), kw=kwargs: make_uniwheel_env(render, **kw))
    return gym.vector.SyncVectorEnv(env_factories)


## Phase 1: Warm-start from stage 02

No domain randomization yet - just load stage 02's trained weights into a fresh
`Agent` sized for this env, and confirm it actually runs (obs/action shapes are
identical between `UniWheelEnv` in stage 02 and here, so this should just work).

In [ ]:
envs = make_envs(NUM_ENVS)  # domain_rand=None by default - identical to stage 02's env

# Work around a quirk in the shared evaluate(): gym.wrappers.RecordEpisodeStatistics doesn't
# forward custom attributes to the wrapped env, so evaluate()'s getattr(eval_envs.envs[0],
# 'max_steps', config.num_steps) always misses and silently truncates eval episodes at
# config.num_steps (2048) instead of the real 10000 - understating returns by ~5x (see
# rl/README.md). Setting it directly on the wrapper instance fixes it without touching the
# shared trainer file.
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.max_steps = env_stat_wrapper.env.max_steps

device = torch.device("cuda" if (torch.cuda.is_available() and ppo_config.cuda) else "cpu")
agent = Agent(envs, ppo_config).to(device)
agent.load_state_dict(torch.load(STAGE02_CHECKPOINT, map_location=device, weights_only=True))
agent.eval()
print(f"Loaded stage 02 checkpoint from {STAGE02_CHECKPOINT}")

result = TrainResult(
    agent=agent,
    checkpoint_dir=STAGE02_CHECKPOINT.parent,
    best_model_path=STAGE02_CHECKPOINT,
    final_model_path=None,
    best_mean_return=0.0,
)


In [ ]:
# Confirm it still balances before adding any randomization
eval_config = replace(ppo_config, timestep=0.005)
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return (no DR, sanity check): {np.nanmean(returns):.2f}")


## Phase 2: Observation noise + action delay

Models real IMU noise and the real latency between choosing an action and it taking
effect (I2C round-trip, motor response lag) - in sim, an action takes effect on the
very next `step()`, which the real robot can't match exactly.

In [ ]:
ppo_config.exp_name = "uniwheel-ppo-dr-phase-2"

dr = DomainRandomConfig(
    pitch_noise_std_dev=0.01,
    roll_noise_std_dev=0.01,
    pitch_rate_noise_std_dev=0.01,
    roll_rate_noise_std_dev=0.01,
    wheel_vel_noise_std_dev=0.1,
    action_delay_steps=1,       # 1 step = 5ms at this model's timestep
    action_delay_random=True,
)
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr


In [ ]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


In [ ]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


## Phase 3: Motor noise + random pushes

Models motor driver noise and real-world bumps/nudges - the policy needs to reject
disturbances it didn't cause itself, not just ones from its own reset().

In [ ]:
ppo_config.exp_name = "uniwheel-ppo-dr-phase-3"

dr.motor_noise_scale = 0.02   # +/-2% motor noise
dr.push_prob = 0.005          # 0.5% chance of a push per step
dr.push_force_max_n = 0.3     # gentle pushes - this robot is ~0.4kg, scale accordingly
                                # if you change the mass materially
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr


In [ ]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


In [ ]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


## Phase 4: Mass + friction randomization

Models manufacturing/assembly variance and different floor surfaces.

In [ ]:
ppo_config.exp_name = "uniwheel-ppo-dr-phase-4"

dr.mass_scale_range = (0.8, 1.2)       # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)   # 50-150% friction variation
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr


In [ ]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


In [ ]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


## Phase 5: Motor gain randomization

Models battery sag and part-to-part motor variance (you have two physical FIT1034
units - they won't be identical).

In [ ]:
ppo_config.exp_name = "uniwheel-ppo-dr-phase-5"

dr.motor_gain_range = (0.6, 1.0)   # 60-100% motor torque per episode
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr


In [ ]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


In [ ]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


## Phase 6: Ridge torque (wheel axle only)

Models the ground wheel's tire seam/ridge briefly catching on the floor - a small
random axle torque. **Not applied to the reaction wheel** - it's fully enclosed and
never touches the ground, so this disturbance can't physically happen to it (see
`uniwheel_env_dr.py`'s module docstring).

In [ ]:
ppo_config.exp_name = "uniwheel-ppo-dr-phase-6"

dr.ridge_prob = 0.05             # 5% chance per step
dr.ridge_torque_max_nm = 0.005   # small - see feasibility_analysis.md for the wheel
                                  # motor's real torque budget this should stay well under
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr


In [ ]:
result = train(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")
if result.best_model_path is not None:
    result.agent.load_state_dict(torch.load(result.best_model_path, weights_only=True))


In [ ]:
returns = evaluate(result.agent, eval_episodes=3, config=eval_config, envs=envs)
print(f"Mean return: {np.nanmean(returns):.2f}")
export_tb_plots_as_csv(result.checkpoint_dir)


## Clean up and export

Same export path as stage 03 expects: `actor.onnx` -> `actor.h` via
`rl/onnx_actor_to_c.py`, ready to drop into `03-sim-to-real/`.

In [ ]:
for idx, env in enumerate(envs.envs):
    env.env.close()


In [ ]:
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)
print(f"Exported {result.checkpoint_dir / 'actor.onnx'}")
print("Next: python3 ../rl/onnx_actor_to_c.py " + str(result.checkpoint_dir / "actor.onnx") +
      " actor.h, then copy actor.h into ../03-sim-to-real/uniwheel_bot/")
